# GPAP — GPU SAT Sampling Benchmark (cross-GPU)

Measure **how fast GPAP draws guaranteed-valid SAT models on *this* GPU vs CMSGen**, the SOTA
fast "uniform-like" CPU sampler.

GPAP runs CMSGen's own algorithm — *random variable → random polarity → unit-propagate* — but
**K trajectories in parallel on the GPU**, discarding conflicted ones (every full, conflict-free
trajectory is a model: all clause `sat_count > 0`). Model-preserving unit-propagation
simplification shrinks the formula so a much larger `K` fits — the throughput lever.

Three measurements on the projected-sampling `sk_` benchmarks (independent support via `c ind`):

1. **Throughput vs CMSGen** — GPAP samples/s ÷ CMSGen samples/s (end-to-end wall), per niche
   instance + geomean.
2. **Quality parity** — 2-wise coverage % (the combinatorial-testing metric); the speed win must
   come at equal coverage.
3. **K-scaling** — sweep `K` on a couple of instances: throughput is VRAM-limited so it scales
   ~linearly with `K`. **This is the A100 story** — an 80 GB GPU sits far up the curve.

**Before running:** `Runtime ▸ Change runtime type ▸ GPU` (T4/L4/A100…). `K` auto-sizes to the
GPU's VRAM — no manual tuning. Results save per-GPU to `benchmarks/colab_sampling_results/`.

## 1. Setup — clone repo + build CMSGen
Torch ships preinstalled on Colab. We add `python-sat`, then build **CMSGen** from source
(meelgroup/cmsgen; no pip wheel) so we have the head-to-head competitor.

In [ ]:
import os, subprocess, sys
from getpass import getpass

REPO_OWNER = "yogevshalmon"
REPO_NAME  = "gpap"
BRANCH     = "gpu_bcp_sampling"   # the sampling-direction branch

# Preferred: store your PAT as a Colab Secret named GITHUB_TOKEN (left sidebar -> key icon).
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN") or ""
except Exception:
    token = ""
if not token:
    print("No GITHUB_TOKEN secret found.")
    print("Create one at GitHub -> Settings -> Developer settings -> Personal access tokens")
    token = getpass("GitHub PAT (leave blank if repo is public): ")

auth     = f"{token}@" if token else ""
repo_url = f"https://{auth}github.com/{REPO_OWNER}/{REPO_NAME}.git"

if os.path.basename(os.getcwd()) != REPO_NAME:
    if not os.path.isdir(REPO_NAME):
        subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, repo_url, REPO_NAME],
                       check=True)
    os.chdir(REPO_NAME)
subprocess.run(["git", "pull", "--ff-only", repo_url, BRANCH], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "python-sat", "tqdm"], check=True)
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print("repo:", os.getcwd(), "| branch:", BRANCH)

In [ ]:
# Build CMSGen (the comparison sampler). Needs cmake + build-essential + zlib.
# Skip this cell to run GPAP-only (the K-scaling + throughput numbers still work).
subprocess.run(["apt-get", "install", "-y", "-q", "cmake", "build-essential", "zlib1g-dev"],
               check=False)
from scripts.colab_sampling_benchmark import build_cmsgen
CMSGEN = build_cmsgen()
print("cmsgen:", CMSGEN or "(build failed — running GPAP-only)")

## 2. Detect the GPU
Confirms which accelerator this run used (recorded in the result file). The A100's large VRAM is
what drives a large auto-`K`.

In [ ]:
import torch
from scripts.colab_sampling_benchmark import run_all, save_results, compare_saved
from scripts.colab_benchmark import gpu_info

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
info = gpu_info(device)
for k, v in info.items():
    print(f"{k:>16}: {v}")
if info.get("total_gb") is None:
    print("\n  No CUDA GPU detected — set Runtime > Change runtime type > GPU and re-run cell 1.")

## 3. Run the benchmark — CircuitSAT sampling (current focus)

`circuits_only=True` runs **only** the CircuitSAT-sampling suite (the "Recurrent CircuitSAT
Sampling" family): **ISCAS-89 + ITC'99 `b`-instances** unrolled over cycle depth `{1,5,10,25}`
(and ISCAS-85 combinational on a full run), each pinned to a reachable output target and sampled
over its primary inputs. Any unroll exceeding 100k clauses is skipped automatically.

`quick=True` finishes in a few minutes (a small circuit subset, smaller sample counts, K-sweep on
the best circuit). Set `quick=False` for the whole set. `K` auto-sizes to all available VRAM, so the
A100 produces a much larger auto-`K` and lands further up the K-scaling curve. The run reports
throughput vs CMSGen **and** a UNIQUE-solutions count (distinct support-projections) computed off the
timing path — confirming the sampler emits genuinely distinct solutions.

In [ ]:
# Circuits-only run (the current focus): ISCAS-89 + ITC'99 b-instances unrolled over cycle depth,
# plus ISCAS-85 combinational on a full run. The adaptive bandit picks a circuit portfolio that adds
# cheap "soft" conflict-avoidance lookahead — the big win on the tight/deep circuits. Also reports a
# UNIQUE-solutions count (distinct support-projections; measured off the throughput timing path).
# For the sk_ suite instead, pass circuits_only=False, do_circuits=False.
payload = run_all(quick=True, cmsgen_bin=CMSGEN, circuits_only=True)

## 4. Save this GPU's result
Writes `benchmarks/colab_sampling_results/<gpu-slug>.json` (one file per GPU; re-runs overwrite).

In [ ]:
path = save_results(payload)

# Optional: download a copy to your machine.
try:
    from google.colab import files
    files.download(path)
except Exception:
    pass

## 4b. Plots
K-scaling (samples/s vs K, with this GPU's auto-K marked), GPAP vs CMSGen throughput, and the
2-wise coverage parity scatter.

In [ ]:
from scripts.colab_sampling_plots import make_plots
from scripts.colab_sampling_benchmark import DEFAULT_OUT_DIR
from IPython.display import Image, display

for p in make_plots(payload, DEFAULT_OUT_DIR):
    display(Image(filename=p))

## 5. (Optional) Commit the result back to the repo
Persist this GPU's numbers in version control so they accumulate across GPUs. Needs a GitHub
**Personal Access Token** with `repo` write scope. Uncomment, fill in, and run:

In [ ]:
# from getpass import getpass
# if not token:  # reuse token from cell 1 if already set
#     token = getpass("GitHub PAT: ")
# GIT_NAME, GIT_EMAIL = "Your Name", "you@example.com"
#
# subprocess.run(["git", "config", "user.name", GIT_NAME], check=True)
# subprocess.run(["git", "config", "user.email", GIT_EMAIL], check=True)
# subprocess.run(["git", "add", "benchmarks/colab_sampling_results/"], check=True)
# subprocess.run(["git", "commit", "-m", f"colab sampling: {payload['gpu']['name']} result"], check=True)
# remote = f"https://{token}@github.com/yogevshalmon/gpap.git"
# subprocess.run(["git", "push", remote, f"HEAD:{BRANCH}"], check=True)
# print("pushed")

## 6. Compare GPUs
Ranks every result JSON currently in `benchmarks/colab_sampling_results/` by peak GPAP throughput
(and shows auto-`K` + geomean vs CMSGen). Re-run cell 1 to pull results committed from other GPUs first.

In [ ]:
_ = compare_saved()